In [2]:
import numpy as np
import matplotlib.pyplot as plt
from galpy.potential import (NFWPotential, TwoPowerSphericalPotential, AnySphericalPotential,evaluateDensities)
from colossus.cosmology import cosmology
from colossus.halo import concentration, profile_nfw
from scipy.optimize import minimize, fsolve
from math import pi
from galpy.orbit import Orbit       
import sys
import pandas as pd
import time
from astropy.constants import G
from astropy import units as u
from galpy.potential import ChandrasekharDynamicalFrictionForce as CDFF, vcirc, FDMDynamicalFrictionForce as FDM_DF
from joblib import Parallel, delayed
from matplotlib.colors import LogNorm
from matplotlib.cm import ScalarMappable
from halotools.sim_manager import FakeSim
from halotools.empirical_models import PrebuiltSubhaloModelFactory
import matplotlib.lines as mlines
from scipy.stats import gaussian_kde
from matplotlib.ticker import AutoMinorLocator
from matplotlib.ticker import MultipleLocator
from scipy.signal import argrelextrema
import random

In [3]:
def zeta(z, Omega_m):
    return (18 * np.pi**2 + 82 * (Omega_m - 1) - 39 * (Omega_m - 1)**2) / Omega_m
def r_c(z, M_h, m_22, Omega_m):
    return (1.6 / np.sqrt(1 + z)) * (zeta(z, Omega_m) / zeta(0, Omega_m0))**(-1/6) * (M_h / 1e9)**(-1/3) * m_22**-1
def rho2_z(r_c, m_22, z):
    return 1.9 * (10 * m_22)**-2 * r_c**-4 * 1e9 / ((1 + z)**3)

zgo = 0
cosmo = cosmology.setCosmology('planck18')
Omega_m0 = cosmo.Om(zgo)

def FDMhalo(m22, M):

    Mh=M*0.704
    rc1 = r_c(zgo, M, m22, Omega_m0)
    rhoFDM = rho2_z(rc1, m22, zgo)
    AmpFDM = rhoFDM * 4 * pi * rc1**3

    rc2 = r_c(zgo, M, m22/10, Omega_m0)
    rhoFDM2 = rho2_z(rc2, m22/10, zgo)
    AmpFDM2 = rhoFDM2 * 4 * pi * rc2**3
    
    C2 = concentration.concentration(M*0.704, 'vir', zgo, model='prada12')
    p_nfw = profile_nfw.NFWProfile(M=M*0.704, c=C2, z=zgo, mdef='vir')
    rs = p_nfw.par['rs'] / 0.704
    A=np.log(1+C2)-C2/(1+C2)

    DMhalo2 = NFWPotential(amp=M/A * u.Msun, a=rs*u.kpc)
    r_vals = np.logspace(np.log10(0.001), np.log10(1000), 200)
    FDMhalo_soliton = AnySphericalPotential(dens=lambda r: rhoFDM / ((1 + 0.091 * (r / rc1)**2)**8))

    def cost(params):
        psi, gamma, eta = params
        if psi <= 0 or gamma <= 0 or eta <= 0:
            return np.inf

        def equation(alpha):
            FDMhalo3 = TwoPowerSphericalPotential(amp=psi * AmpFDM2 * (alpha**3) * u.Msun, a=alpha * rc2 * u.kpc, alpha=0, beta=3)
            dens_FDM3 = evaluateDensities(FDMhalo3, 100 * u.kpc, 0 * u.kpc)
            dens_CDM = evaluateDensities(DMhalo2, 100 * u.kpc, 0 * u.kpc)
            return (dens_FDM3 - dens_CDM) / dens_CDM

        try:
            alpha_sol = fsolve(equation, 10)[0]
        except:
            return np.inf

        FDMhalo3 = TwoPowerSphericalPotential(amp=psi * AmpFDM2 * (alpha_sol**3) * u.Msun, a=alpha_sol * rc2 * u.kpc, alpha=0, beta=3)
        CoreFDM = TwoPowerSphericalPotential(amp=AmpFDM * (gamma**3) * u.Msun, a=gamma * rc1 * u.kpc, alpha=0, beta=eta)
        total_ref = np.array([evaluateDensities(FDMhalo_soliton, r, 0) + evaluateDensities(FDMhalo3, r * u.kpc, 0 * u.kpc) * 1e9 for r in r_vals])
        total_test = np.array([evaluateDensities(CoreFDM, r * u.kpc, 0 * u.kpc) * 1e9 + evaluateDensities(FDMhalo3, r * u.kpc, 0 * u.kpc) * 1e9 for r in r_vals])
        return np.mean((np.log10(total_ref + 1e-30) - np.log10(total_test + 1e-30))**2)

    result = minimize(cost, x0=[0.5, 12.0, 20.0], bounds=[(0.0001, 0.7 * (1e10 / M) * (10 / m22)), (1, 10), (3, 10)], method='L-BFGS-B')
    psi_opt, gamma_opt, eta_opt = result.x

    def equation(alpha):
        FDMhalo3 = TwoPowerSphericalPotential(amp=psi_opt * AmpFDM2 * (alpha**3) * u.Msun, a=alpha * rc2 * u.kpc, alpha=0, beta=3)
        dens_FDM3 = evaluateDensities(FDMhalo3, 100 * u.kpc, 0 * u.kpc) * 1e9
        dens_CDM = evaluateDensities(DMhalo2, 100 * u.kpc, 0 * u.kpc) * 1e9
        return (dens_FDM3 - dens_CDM) / dens_CDM

    alpha_final = fsolve(equation, 10)[0]
    FDMhalo3 = TwoPowerSphericalPotential(amp=psi_opt * AmpFDM2 * (alpha_final**3) * u.Msun, a=alpha_final * rc2 * u.kpc, alpha=0, beta=3)
    CoreFDM = TwoPowerSphericalPotential(amp=(gamma_opt**3) * AmpFDM * u.Msun, a=gamma_opt * rc1 * u.kpc, alpha=0, beta=eta_opt)
    return CoreFDM + FDMhalo3



        #tri des données euclid

def mass_from_mag(MAG_GC):
    return 1.8 * 10**(-0.4 * (MAG_GC + 0.5 - 4.83))

def count_unique(arr):
    return len(set(arr))

Rlimit=0.2 #kpc
MAGlimit = -8

file_path = "ERO_Fornax_GCs_results_for_Pierre_Feb2026.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

logMs=np.log10(df["logMstar_host"].to_numpy())   #fornax
#logMs=df["logMstar_host"].to_numpy()         #perseus
IDgal=df["gal_name"].to_numpy()
MAG_GC=df["GC_VIS_abs"].to_numpy()
RGC=df["GC_dist_kpc"].to_numpy()
print("total number of GCs :",len(RGC))
print('total number of galaxies :',count_unique(IDgal))


sola=np.where(MAG_GC<=MAGlimit)[0]
logMs=logMs[sola]
RGC=RGC[sola]
MAG_GC=MAG_GC[sola]
IDgal=IDgal[sola]

sol=np.where(RGC>Rlimit)[0]
logMs=logMs[sol]
MAG_GC=MAG_GC[sol]
RGC=RGC[sol]
IDgal=IDgal[sol]
MGC=mass_from_mag(MAG_GC)

print("final number of GCs ",len(RGC))
print('final number of galaxies (contening GCs):',count_unique(IDgal))
df_clean = pd.DataFrame({
    "galaxy": IDgal,
    "R": RGC,
    "MAG": MAG_GC,
    "logMs": logMs})

groups = df_clean.groupby("galaxy")

total number of GCs : 80
total number of galaxies : 18
final number of GCs  39
final number of galaxies (contening GCs): 12


In [7]:
ts=np.linspace(0,10,1001)*u.Gyr 
rhm=0.01 #kpc 
halocat=FakeSim() 
model=PrebuiltSubhaloModelFactory('behroozi10') 
model.populate_mock(halocat)

keep_rows = []

for name, g in groups:
    R = g["R"].to_numpy()
    MAG = g["MAG"].to_numpy()
    logM = g["logMs"].iloc[0]
    M = mass_from_mag(MAG)

    Mg = 10**model.mean_log_halo_mass(logM)
    Mh = Mg * 0.704

    c = concentration.concentration(Mh,'vir',0,model='prada12')
    p_nfw = profile_nfw.NFWProfile(M=Mh,c=c,z=0,mdef='vir')

    rs = p_nfw.par['rs']/0.704
    A = np.log(1+c) - c/(1+c)

    NFW_pot = NFWPotential(amp=Mg/A*u.Msun, a=rs*u.kpc)

    for i in range(len(R)):

        orb = Orbit([R[i]*u.kpc, 0*u.km/u.s, vcirc(NFW_pot, R[i]*u.kpc)*u.km/u.s, 0*u.kpc, 0*u.km/u.s, 0*u.deg ])
        cdf = CDFF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc, dens=NFW_pot)
        orb.integrate(ts, NFW_pot + cdf, method='dop853_c')
        if np.min(orb.r(ts)) >= Rlimit:
            keep_rows.append(g.iloc[i])

df_2 = pd.DataFrame(keep_rows)
print(df_2)

             galaxy         R        MAG     logMs
20  FORNAX-DWARF-10  5.607620  -8.970728  7.463588
21  FORNAX-DWARF-10  2.041360  -8.219956  7.463588
22  FORNAX-DWARF-10  5.914274  -8.473175  7.463588
23  FORNAX-DWARF-10  1.587830  -9.335814  7.463588
24  FORNAX-DWARF-10  3.332258  -9.064346  7.463588
26  FORNAX-DWARF-11  1.990622  -9.082847  7.194519
27  FORNAX-DWARF-11  4.750817  -8.221331  7.194519
28  FORNAX-DWARF-13  2.479623  -8.262112  7.013184
30  FORNAX-DWARF-13  3.716957  -9.318089  7.013184
31  FORNAX-DWARF-13  2.670599  -8.567513  7.013184
32  FORNAX-DWARF-13  1.791642  -8.679402  7.013184
33  FORNAX-DWARF-13  3.249625  -8.963624  7.013184
34  FORNAX-DWARF-13  3.050812  -8.415562  7.013184
35  FORNAX-DWARF-18  1.431819  -8.679487  7.061677
37  FORNAX-DWARF-28  2.077203  -8.206694  5.775802
0    FORNAX-DWARF-3  1.687586  -9.245816  8.344921
1    FORNAX-DWARF-3  1.243947  -9.426696  8.344921
2    FORNAX-DWARF-3  2.175778  -8.449199  8.344921
3    FORNAX-DWARF-3  1.432765  

In [ ]:
sample = df_2.sample(n=3, replace=False)
RGC = []
M= []
logMs = []

for _, row in sample.iterrows():
    R.append(row["R"])

    M = mass_from_mag(np.array([row["MAG"]]))[0]
    MGC.append(M)

    logMS_list.append(row["logMs"])
    